Stage1



In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 15.4 MB/s eta 0:00:00


Hugging Face login

In [ ]:
from huggingface_hub import login
login()


Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


Load Gemma model (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 🔴 Correct 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (n

In [ ]:
!pip install -q bitsandbytes accelerate transformers peft

— Upload math book PDF

In [ ]:
from google.colab import files

uploaded = files.upload()  # Upload math_book.pdf

Convert PDF → text

In [ ]:
from pypdf import PdfReader

reader = PdfReader("math_book.pdf")

raw_text = []
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text.append(text)

full_text = "\n".join(raw_text)

print("PDF extracted")

In [ ]:
clean_lines = []

for line in full_text.split("\n"):
    line = line.strip()

    if len(line) < 40:
        continue
    if "Answer" in line or "Solution" in line:
        continue
    if any(char.isdigit() for char in line) and "=" in line:
        continue

    clean_lines.append(line)

print("Clean lines:", len(clean_lines))

Create dataset

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": clean_lines
})

dataset

Tokenize dataset

In [ ]:
def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # \ THIS LINE FIXES YOUR ERROR
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_ds = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#  VERY IMPORTANT STEP

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # works for Gemma
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Training configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-math-domain",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=200,
    save_total_limit=1,
    report_to="none"
)

Train (Math Book Adaptation)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds
)

trainer.train()

In [ ]:
model.save_pretrained("gemma-math-domain-lora")
tokenizer.save_pretrained("gemma-math-domain-lora")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("gemma-math-domain-lora", "zip", "gemma-math-domain-lora")
files.download("gemma-math-domain-lora.zip")
print("Download started!")

Light sanity check

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 🔹 Base model
model_name = "google/gemma-2b"

# 🔹 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 🔹 4-bit quantization config (correct way for Gemma)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 🔹 Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# 🔹 Load LOCAL LoRA adapters (FIXED PATH)
model = PeftModel.from_pretrained(
    model,
    "./gemma-math-domain-lora"
)

# 🔹 Merge adapters into base model (recommended for inference)
model = model.merge_and_unload()

model.eval()

# 🔹 Sanity test prompt
prompt = "Explain what a mathematical function is."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 🔹 Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

# 🔹 Print result
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import gc
import torch

try:
    del model
except:
    pass

try:
    del tokenizer
except:
    pass

torch.cuda.empty_cache()
gc.collect()

print("Cleanup completed")

In [ ]:
# Enable training for LoRA
model.enable_input_require_grads()
model.print_trainable_parameters()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("gemma-math-domain-lora.zip", 'r') as zip_ref:
    zip_ref.extractall("gemma-math-domain-lora")

In [ ]:
## Load Stage-1 LoRA adapters
model = PeftModel.from_pretrained(
    model,
   "./gemma-math-domain-lora"
)

# 🔴 ADD THIS PART (Stage-2 LoRA)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Check trainable params
model.print_trainable_parameters()

# Train mode
model.train()

 **Stage 2
**


Generate base dataset:

Load into HuggingFace Dataset

**----------------------------------------------------------------------------------------------------------------------------------------------------------------**

Domain specific fine tuning:

Install and extract from both PDFs:

In [ ]:
!pip install pypdf -q

from pypdf import PdfReader
import re, json, random

# Extract math book text
math_reader = PdfReader("math_book.pdf")
math_text = ""
for page in math_reader.pages:
    math_text += page.extract_text() + "\n"

# Extract history book text
hist_reader = PdfReader("History-Class-7th.pdf")
hist_text = ""
for page in hist_reader.pages:
    hist_text += page.extract_text() + "\n"

print("Math text length:", len(math_text))
print("History text length:", len(hist_text))

Extract real questions from both books:

In [ ]:
def extract_math_questions(text):
    lines = text.split('\n')
    questions = []
    for line in lines:
        line = line.strip()
        # Numbered exercise questions with enough content
        if re.match(r'^\d+\.', line) and len(line) > 25 and len(line) < 250:
            # Skip chapter headings and section titles
            if not any(skip in line for skip in ["Chapter", "EXERCISE", "Fig", "NCERT"]):
                questions.append(line)
    return list(set(questions))  # remove duplicates

def extract_history_questions(text):
    lines = text.split('\n')
    questions = []
    for line in lines:
        line = line.strip()
        if '?' in line and len(line) > 25 and len(line) < 250:
            # Only keep lines that start with question words or numbers
            if re.match(r'^(\d+\.|What|Who|How|Why|When|Where|Describe|Explain|Discuss|Analyse)', line):
                questions.append(line)
    return list(set(questions))

math_questions = extract_math_questions(math_text)
history_questions = extract_history_questions(hist_text)

print(f"Math questions extracted: {len(math_questions)}")
print(f"History questions extracted: {len(history_questions)}")
print("\nSample math:")
for q in math_questions[:5]:
    print(" ", q)
print("\nSample history:")
for q in history_questions[:5]:
    print(" ", q)

Build harder/easier pairs from real textbook questions:

In [ ]:
# Math harder/easier transformation rules
def make_math_harder(q):
    # Add a constraint or extra operation
    templates = [
        f"Solve the following and verify your answer: {q}",
        f"{q} Also find the value if the result is doubled.",
        f"{q} Express your answer as a fraction and a decimal.",
    ]
    return random.choice(templates)

def make_math_easier(q):
    # Simplify by breaking into a hint
    templates = [
        f"Hint: Start by identifying the numbers. {q}",
        f"Fill in the blank to solve: {q}",
        f"Use a number line to solve: {q}",
    ]
    return random.choice(templates)

def make_history_harder(q):
    q_clean = q.lstrip("0123456789. ").strip()
    templates = [
        f"Critically analyze: {q_clean} Support your answer with examples.",
        f"With reference to medieval India, {q_clean.lower()} Discuss in detail.",
        f"Examine the causes and effects: {q_clean}",
    ]
    return random.choice(templates)

def make_history_easier(q):
    q_clean = q.lstrip("0123456789. ").strip()
    # Strip it to the core subject
    words = q_clean.split()
    subject = " ".join(words[:6])
    templates = [
        f"In one sentence, answer: {q_clean}",
        f"Name one example of: {subject}",
        f"True or False: {subject}",
    ]
    return random.choice(templates)

# Build dataset
dataset = []

# Math pairs
for q in math_questions:
    # Harder
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": q,
        "output": make_math_harder(q),
        "direction": "harder",
        "subject": "math",
        "concept": "textbook_exercise"
    })
    # Easier
    dataset.append({
        "instruction": "Rewrite the question to easier difficulty while keeping the same concept.",
        "input": q,
        "output": make_math_easier(q),
        "direction": "easier",
        "subject": "math",
        "concept": "textbook_exercise"
    })

# History pairs
for q in history_questions:
    # Harder
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": q,
        "output": make_history_harder(q),
        "direction": "harder",
        "subject": "history",
        "concept": "textbook_conceptual"
    })
    # Easier
    dataset.append({
        "instruction": "Rewrite the question to easier difficulty while keeping the same concept.",
        "input": q,
        "output": make_history_easier(q),
        "direction": "easier",
        "subject": "history",
        "concept": "textbook_conceptual"
    })

random.shuffle(dataset)
print(f"Total dataset size: {len(dataset)}")
print(f"Math harder: {sum(1 for d in dataset if d['subject']=='math' and d['direction']=='harder')}")
print(f"Math easier: {sum(1 for d in dataset if d['subject']=='math' and d['direction']=='easier')}")
print(f"History harder: {sum(1 for d in dataset if d['subject']=='history' and d['direction']=='harder')}")
print(f"History easier: {sum(1 for d in dataset if d['subject']=='history' and d['direction']=='easier')}")

print("\nSample harder math:")
print(next(d for d in dataset if d['subject']=='math' and d['direction']=='harder'))
print("\nSample easier history:")
print(next(d for d in dataset if d['subject']=='history' and d['direction']=='easier'))

Generate and download new 8000 dataset:

In [ ]:
import json, random
from google.colab import files

dataset = []

names = ["Sara","Aman","Priya","Rahul","Arjun","Maya","John","Riya"]
objects = ["books","pens","marbles","apples","stickers","notebooks"]

def generate_linear():
    a = random.randint(2,8)
    b = random.randint(1,9)
    c = random.randint(10,40)
    base = f"Solve {a}x + {b} = {c}"
    harder = f"Solve {a}(2x - {b}) + {b+3} = {2*c - b + 3}"
    return base, harder

def generate_counting():
    name = random.choice(names)
    obj = random.choice(objects)
    n1 = random.randint(3,15)
    n2 = random.randint(2,10)
    base = f"{name} has {n1} {obj}. {name} buys {n2} more. How many {obj} does {name} have now?"
    harder = f"{name} has {n1} {obj}. {name} buys {n2} more, gives 3 away, and later buys 4 more. How many {obj} does {name} finally have?"
    return base, harder

history = [
    ("Who was Akbar?",
     "Explain the major administrative reforms introduced by Akbar during the Mughal Empire."),
    ("What was the Delhi Sultanate?",
     "Describe the origin and political significance of the Delhi Sultanate in medieval India."),
    ("What was the Bhakti movement?",
     "Analyze the social and religious impact of the Bhakti movement in medieval India."),
    ("What were medieval Indian trading towns?",
     "Explain the economic importance of trading towns in medieval India."),
    ("What is a monument?",
     "Discuss how monuments help historians understand medieval Indian architecture.")
]

# Generate base harder examples
for _ in range(1500):
    inp, out = generate_linear()
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": inp, "output": out,
        "subject": "math", "concept": "linear_equation"
    })

for _ in range(1500):
    inp, out = generate_counting()
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": inp, "output": out,
        "subject": "math", "concept": "counting_problem"
    })

for _ in range(1000):
    q = random.choice(history)
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": q[0], "output": q[1],
        "subject": "history", "concept": "conceptual"
    })

# Now double it by adding easier examples (swap input/output)
full_dataset = []
for example in dataset:
    # Harder entry
    full_dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": example["input"],
        "output": example["output"],
        "direction": "harder",
        "subject": example["subject"],
        "concept": example["concept"]
    })
    # Easier entry (swap)
    full_dataset.append({
        "instruction": "Rewrite the question to easier difficulty while keeping the same concept.",
        "input": example["output"],
        "output": example["input"],
        "direction": "easier",
        "subject": example["subject"],
        "concept": example["concept"]
    })

random.shuffle(full_dataset)

with open("difficulty_transformation_dataset_8000.json", "w") as f:
    json.dump(full_dataset, f, indent=2)

print("Dataset saved!")
print("Total samples:", len(full_dataset))
print("Harder:", sum(1 for d in full_dataset if d["direction"] == "harder"))
print("Easier:", sum(1 for d in full_dataset if d["direction"] == "easier"))

files.download("difficulty_transformation_dataset_8000.json")

Combine with original synthetic dataset and save:

In [ ]:
# Load original 8000 dataset
with open("difficulty_transformation_dataset_8000.json") as f:
    original_data = json.load(f)

# Combine both
combined = original_data + dataset
random.shuffle(combined)

with open("difficulty_dataset_combined.json", "w") as f:
    json.dump(combined, f, indent=2)

print("Combined dataset saved!")
print("Original samples:", len(original_data))
print("Textbook samples:", len(dataset))
print("Total combined:", len(combined))

Download

In [ ]:
from google.colab import files
files.download("difficulty_dataset_combined.json")

Upload the downloaded dataset back to Colab:

In [ ]:
from google.colab import files
import json

uploaded = files.upload()

with open("difficulty_transformation_dataset_8000.json") as f:
    data = json.load(f)

print("Loaded samples:", len(data))
print("Sample entry:")
print(data[0])

 Load into HuggingFace Dataset:

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(data)
print(dataset)

Format dataset:

In [ ]:
def format_example(example):

    text = f"""{example['instruction']}

Question:
{example['input']}

Rewritten Question:
{example['output']}"""

    return {"text": text}

formatted_dataset = dataset.map(format_example)

formatted_dataset = formatted_dataset.remove_columns(
    ["instruction","input","output","subject","concept"]
)

print(formatted_dataset[0])

Reload tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Tokenize

In [ ]:
tokenized_dataset = formatted_dataset.map(tokenize)

 Tokenization:

In [ ]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = formatted_dataset.map(tokenize)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

print(tokenized_dataset)

Training config:

In [ ]:
## Clearing GPU
import torch, gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-stage2",

    num_train_epochs=3,   # reduce from 5 → 3 (major time save)

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # reduce from 8 → faster training

    learning_rate=2e-4,

    logging_steps=20,
    save_strategy="epoch",

    fp16=True,
    optim="paged_adamw_8bit",

    report_to="none"
)

Trainer and train:

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
import torch, gc

torch.cuda.empty_cache()
gc.collect()

trainer.train()

In [ ]:
model = PeftModel.from_pretrained(
    model,
   "./gemma-math-domain-lora"
)

# IMPORTANT FIX
model.enable_input_require_grads()
model.print_trainable_parameters()

model.train()

Save Trained Model:

In [ ]:
model.save_pretrained("difficulty_model_stage2_final")
tokenizer.save_pretrained("difficulty_model_stage2_final")
print("Model saved!")

 Zip and download trained model:

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("difficulty_model_stage2_final", "zip", "difficulty_model_stage2_final")
files.download("difficulty_model_stage2_final.zip")
print("Download started!")

Import Libraries

Part 2 ---------------------------------------------------------------------------------------------------------------------------------


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

Login to HuggingFace

In [ ]:
from huggingface_hub import login
login()

Load the base Gemma model (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
!pip uninstall -y sympy
!pip install sympy

Import saved trained model

In [ ]:
from google.colab import files
files.upload()

Unzip

In [ ]:
import zipfile

with zipfile.ZipFile("stage2_gemma_lora.zip", 'r') as zip_ref:
    zip_ref.extractall("stage2_gemma_lora")

load stage1 LoRA adapters

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_name = "google/gemma-2b"

# 🔹 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 🔹 4-bit config (correct for Gemma)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 🔹 Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

# 🔹 Load Stage-1 LoRA adapters (IMPORTANT)
model = PeftModel.from_pretrained(
    model,
    "./gemma-math-domain-lora"
)

# 🔹 Keep in training mode (for Stage 2)
model.train()

Load Trained LoRA

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "stage2_gemma_lora"
)

model.eval()

Testing Prompts

In [ ]:
import re

The rewrite function:

In [ ]:
def rewrite_question(question, direction="harder", q_type="math"):

    if q_type == "math":
        if direction == "harder":
            prompt = f"""Task: Make this equation harder by adding brackets and more terms.

Input: 3x - 1 = 8
Output: 4(3x - 1) + 2 = 2x + 15

Input: {question}
Output:"""

        else:
            prompt = f"""Task: Simplify this equation by removing brackets and reducing terms.

Input: 4(3x - 1) + 2 = 2x + 15
Output: 3x - 1 = 8

Input: {question}
Output:"""

    elif q_type == "counting":
        if direction == "harder":
            prompt = f"""Task: Take the exact question below and add more steps to make it harder.

Input: Tom has 4 apples. He buys 2 more. How many apples does he have?
Output: Tom has 4 apples. He buys 2 more, gives 3 away, then buys 5 more. How many apples does he have?

Input: {question}
Output:"""

        else:
            prompt = f"""Task: Take the exact question below and remove steps to make it simpler.

Input: Tom has 4 apples. He buys 2 more, gives 3 away, then buys 5 more. How many apples does he have?
Output: Tom has 4 apples. He buys 2 more. How many apples does he have?

Input: {question}
Output:"""

    elif q_type == "history":
        if direction == "harder":
            prompt = f"""Task: Rewrite the question below to require more detailed analysis.

Input: Who was Gandhi?
Output: How did Gandhi use non-violent resistance to lead India's independence movement?

Input: {question}
Output:"""

        else:
            prompt = f"""Task: Rewrite the question below into a simpler factual question.

Input: How did Gandhi use non-violent resistance to lead India's independence movement?
Output: Who was Gandhi?

Input: {question}
Output:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        repetition_penalty=1.8,
        eos_token_id=tokenizer.encode("\n")[0]
    )

    full_output = tokenizer.decode(output[0], skip_special_tokens=True)
    rewritten = full_output.split("Output:")[-1].strip()
    rewritten = re.split(r'[<>]', rewritten)[0].strip()
    rewritten = re.split(r'\n', rewritten)[0].strip()

    for stop in ["Input", "Example", "Task", "Rewritten", "REWRI",
                 "REWORK", "Question", "###", "</s>", "code"]:
        rewritten = rewritten.split(stop)[0].strip()

    return rewritten

Rule based counting rewriter

In [ ]:
def rewrite_counting(question, direction="harder"):
    sentences = [s.strip() for s in question.strip().split('.') if s.strip()]
    first = sentences[0]
    question_part = sentences[-1] if '?' in sentences[-1] else "How many does she have now"
    question_part = question_part.replace('?', '').strip()

    if direction == "harder":
        return first + ". She buys 5 more, gives 2 away, and then receives 3 from a friend. " + question_part + "?"
    else:
        return first + ". " + question_part + "?"


def rewrite_math_harder(question):
    eq = question.replace("Solve", "").strip()
    if "=" in eq:
        left, right = eq.split("=")
        left = left.strip()
        right = right.strip()
        return f"3({left}) + 5 = 2x + {right}"
    return question


def rewrite_history_harder(question):
    for keyword in ["Who was", "What was", "Who is", "What is"]:
        if question.startswith(keyword):
            subject = question.replace(keyword, "").strip().strip("?").strip()
            return f"How did {subject} influence the political and administrative developments of their time?"
    return "Analyze the long-term impact of " + question.strip("?").lower() + " on historical developments."


def rewrite_history_easier(question):
    for keyword in ["How did", "How was", "Explain how", "Explain the",
                    "Describe how", "Describe the", "What were", "What was"]:
        if question.startswith(keyword):
            words = question.replace(keyword, "").strip().split()
            subject = " ".join(words[:3]).strip("'s").strip(",").strip()
            return f"Who was {subject}?"
    return question

Prompt-based rewriter for math and history:

In [ ]:
def rewrite_question(question, direction="harder", q_type="math"):

    if q_type == "counting":
        return rewrite_counting(question, direction)

    elif q_type == "history":
        if direction == "harder":
            return rewrite_history_harder(question)
        else:
            return rewrite_history_easier(question)

    elif q_type == "math":
        if direction == "harder":
            return rewrite_math_harder(question)

        else:
            prompt = f"""Task: Rewrite by removing all brackets. Keep only one variable term on the left side.
Difficulty: HARD → EASY

Input: 4(2x + 1) - 3 = 3x + 10
Output: 2x = 8

Input: 3(x + 2) - 4 = 2x + 7
Output: x = 3

Input: {question}
Output:"""

            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            output = model.generate(
                **inputs,
                max_new_tokens=60,
                do_sample=False,
                repetition_penalty=1.8,
                eos_token_id=tokenizer.encode("\n")[0]
            )

            full_output = tokenizer.decode(output[0], skip_special_tokens=True)
            rewritten = full_output.split("Output:")[-1].strip()
            rewritten = re.split(r'[<>]', rewritten)[0].strip()
            rewritten = re.split(r'\n', rewritten)[0].strip()

            for stop in ["Input", "Example", "Task", "Difficulty",
                         "Rewritten", "REWRI", "REFORM", "REWORK",
                         "Question", "###", "</s>", "code"]:
                rewritten = rewritten.split(stop)[0].strip()

            return rewritten

Test math:

In [ ]:
question = "Solve 2x + 3 = 7"

print("Harder:", rewrite_question(question, direction="harder", q_type="math"))
print("Easier:", rewrite_question(question, direction="easier", q_type="math"))

Test counting:

In [ ]:
question = "Sara has 7 books. She buys 3 more. How many books does she have?"

print("Harder:", rewrite_question(question, direction="harder", q_type="counting"))
print("Easier:", rewrite_question(question, direction="easier", q_type="counting"))

Test History:

In [ ]:
question = "Who was Akbar?"

print("Harder:", rewrite_question(question, direction="harder", q_type="history"))
print("Easier:", rewrite_question(question, direction="easier", q_type="history"))

Experimenting with interactive interface for adjusting user's question difficulty

In [ ]:
def rewrite_counting(question, direction="harder"):
    sentences = [s.strip() for s in question.strip().split('.') if s.strip()]
    first = sentences[0]

    # Fix question part — find sentence with ?
    question_part = ""
    for s in sentences:
        if '?' in s:
            question_part = s.replace('?', '').replace('"', '').strip()
            break
    if not question_part:
        question_part = "How many does she have now"

    if direction == "harder":
        middle = " ".join(sentences[1:-1]) if len(sentences) > 2 else sentences[1] if len(sentences) > 1 else ""
        if middle:
            return first + ". " + middle + ". She then buys 4 more and gives 2 away. " + question_part + "?"
        else:
            return first + ". She then buys 4 more, gives 2 away, and receives 3 from a friend. " + question_part + "?"
    else:
        return first + ". " + question_part + "?"


def rewrite_math_harder(question):
    eq = question.replace("Solve", "").strip()
    if "=" in eq:
        left, right = eq.split("=")
        left = left.strip()
        right = right.strip()
        return f"3({left}) + 5 = 2x + {right}"
    return question


def rewrite_math_easier(question):
    eq = question.replace("Solve", "").strip()
    if "=" in eq:
        left, right = eq.split("=")
        left = left.strip()
        right = right.strip()
        # Remove brackets if any
        left = re.sub(r'[()]', '', left).strip()
        # Keep only the first term with variable
        terms = left.split('+')
        first_term = terms[0].strip()
        return f"Solve {first_term} = {right}"
    return question


def rewrite_history_harder(question):
    for keyword in ["Who was", "What was", "Who is", "What is"]:
        if question.startswith(keyword):
            subject = question.replace(keyword, "").strip().strip("?").strip()
            return f"How did {subject} influence the political and administrative developments of their time?"
    return "Analyze the long-term impact of " + question.strip("?").lower() + " on historical developments."


def rewrite_history_easier(question):
    # If already a simple question, return simpler version
    for keyword in ["Who was", "What was", "Who is", "What is"]:
        if question.startswith(keyword):
            subject = question.replace(keyword, "").strip().strip("?").strip()
            # Already simple — make it even simpler fill in the blank
            return f"_______ was a famous historical figure."

    for keyword in ["How did", "How was", "Explain how", "Explain the",
                    "Describe how", "Describe the", "What were", "What was"]:
        if question.startswith(keyword):
            words = question.replace(keyword, "").strip().split()
            subject = " ".join(words[:3]).strip("'s").strip(",").strip()
            return f"Who was {subject}?"

    return "What is one fact about " + question.strip("?").lower().replace("who was", "").replace("what was", "").strip() + "?"

Main function:

In [ ]:
def rewrite_question(question, direction="harder", q_type="math"):

    if q_type == "counting":
        return rewrite_counting(question, direction)

    elif q_type == "history":
        if direction == "harder":
            return rewrite_history_harder(question)
        else:
            return rewrite_history_easier(question)

    elif q_type == "math":
        if direction == "harder":
            return rewrite_math_harder(question)
        else:
            return rewrite_math_easier(question)

Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

title = widgets.HTML("<h2>Question Difficulty Rewriter</h2>")

question_input = widgets.Textarea(
    placeholder="Type your question here...",
    layout=widgets.Layout(width="600px", height="100px")
)

q_type_dropdown = widgets.Dropdown(
    options=["math", "counting", "history"],
    description="Type:",
    layout=widgets.Layout(width="250px")
)

direction_toggle = widgets.ToggleButtons(
    options=["harder", "easier"],
    description="Difficulty:",
    button_style="info"
)

run_button = widgets.Button(
    description="Rewrite Question",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px")
)

output_box = widgets.Output()

def on_click(b):
    with output_box:
        clear_output()
        question = question_input.value.strip()
        direction = direction_toggle.value
        q_type = q_type_dropdown.value

        if not question:
            print("Please enter a question first!")
            return

        print(f"Original:   {question}")
        print(f"Direction:  {direction}")
        print(f"Type:       {q_type}")
        print("-" * 50)

        result = rewrite_question(question, direction=direction, q_type=q_type)
        print(f"Rewritten:  {result}")

run_button.on_click(on_click)

display(title)
display(question_input)
display(widgets.HBox([q_type_dropdown, direction_toggle]))
display(run_button)
display(output_box)

##Replacing the rule based pipleine and testing for new examples from each domain.

## Imports & constants

In [ ]:
import gc, json, random, re, zipfile
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
random.seed(42)

BASE_MODEL      = "google/gemma-2b"
STAGE1_ZIP      = "stage2_gemma_lora.zip"
STAGE1_DIR      = Path("stage1_gemma_lora")
STAGE2_ADAPTER  = Path("stage2_adapter")
MAX_LEN         = 384

SYSTEM_PROMPT = (
    "You are a helpful curriculum assistant. "
    "Rewrite the given question to the requested difficulty level "
    "while keeping the topic and subject matter identical. "
    "Return only the rewritten question, nothing else."
)

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

print("Setup complete.")

##Unzip Stage 1 LoRA

In [ ]:
from google.colab import files

# Upload both zips
files.upload()  # upload stage2_gemma_lora.zip and gemma-math-domain-lora.zip

if not STAGE1_DIR.exists():
    with zipfile.ZipFile(STAGE1_ZIP, "r") as z:
        z.extractall(STAGE1_DIR)
    print(f"Unzipped to {STAGE1_DIR}")
else:
    print(f"Already exists: {STAGE1_DIR}")

print("Contents:", [p.name for p in STAGE1_DIR.iterdir()])

##Load base model + Stage 1 LoRA

In [ ]:
from huggingface_hub import login
login()

In [ ]:
def load_base_model():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map="auto",
    )
    return base, tokenizer

clear_memory()
base_model, tokenizer = load_base_model()

# Load stage1 adapter on top
stage1_model = PeftModel.from_pretrained(base_model, str(STAGE1_DIR))
stage1_model = prepare_model_for_kbit_training(stage1_model)
stage1_model.gradient_checkpointing_enable()
stage1_model.config.use_cache = False

print("Stage 1 model loaded.")
stage1_model.print_trainable_parameters()

##Define new training examples

In [ ]:
from google.colab import files

# Upload train.json
files.upload()

with open("train.json") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data)} training examples")

# Build NEW_EXAMPLES from train.json
NEW_EXAMPLES = []
for entry in train_data:
    # Use 'harder' key — fall back to 'medium' if harder not present
    harder_q = entry.get("harder") or entry.get("medium")
    easier_q = entry.get("easy")

    if not harder_q or not easier_q:
        continue  # skip incomplete entries

    NEW_EXAMPLES.append({
        "topic": entry["topic"],
        "base_question": entry["base_question"],
        "harder": harder_q,
        "easier": easier_q,
    })

print(f"Valid training pairs: {len(NEW_EXAMPLES)}")

# Show breakdown by topic
from collections import Counter
topics = Counter(e["topic"] for e in NEW_EXAMPLES)
print("Topics:", dict(topics))
print("\nSample entry:")
print(NEW_EXAMPLES[0])

##Format and tokenize with prompt masking

In [ ]:
def build_prompt_and_full_text(example, difficulty):
    """Returns (prompt_only, prompt+response) for one difficulty direction."""
    prompt = (
        f"### System:\n{SYSTEM_PROMPT}\n\n"
        f"### Input:\n"
        f"Topic: {example['topic']}\n"
        f"Original Question: {example['base_question']}\n"
        f"Target Difficulty: {difficulty}\n\n"
        f"### Response:\n"
    )
    full_text = prompt + example[difficulty]
    return prompt, full_text


def make_instruction_records(examples):
    records = []
    for ex in examples:
        for difficulty in ("harder", "easier"):
            prompt, full_text = build_prompt_and_full_text(ex, difficulty)
            records.append({"prompt": prompt, "full_text": full_text})
    return records


def tokenize_with_masking(example):
    """Mask prompt tokens so the model only trains to generate the answer."""
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )["input_ids"]

    encoded = tokenizer(
        example["full_text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

    labels = encoded["input_ids"].copy()
    prompt_len = min(len(prompt_ids), MAX_LEN)
    # Mask prompt tokens — model does NOT get credit for predicting them
    labels[:prompt_len] = [-100] * prompt_len
    # Mask padding tokens too
    labels = [
        t if m == 1 else -100
        for t, m in zip(labels, encoded["attention_mask"])
    ]
    encoded["labels"] = labels
    return encoded


records = make_instruction_records(NEW_EXAMPLES)
raw_ds  = Dataset.from_list(records)

tokenized_ds = raw_ds.map(
    tokenize_with_masking,
    remove_columns=raw_ds.column_names,
)

print(f"Tokenized dataset: {tokenized_ds}")
print(f"Example — non-masked label tokens: "
      f"{sum(1 for l in tokenized_ds[0]['labels'] if l != -100)}")

##Add Stage 2 LoRA and train

In [ ]:
clear_memory()

# Add a fresh Stage 2 LoRA on top of the Stage 1 model
lora2_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

stage2_model = get_peft_model(stage1_model, lora2_config)
stage2_model.print_trainable_parameters()
stage2_model.train()

stage2_args = TrainingArguments(
    output_dir="./stage2_runs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=6,          # increase if you add more examples
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=tokenized_ds,
)

trainer.train()
print("Stage 2 training complete.")

##Save Stage 2 adapter

In [ ]:
STAGE2_ADAPTER.mkdir(parents=True, exist_ok=True)
stage2_model.save_pretrained(str(STAGE2_ADAPTER))
tokenizer.save_pretrained(str(STAGE2_ADAPTER))
print("Saved to:", STAGE2_ADAPTER)
print("Files:", [p.name for p in STAGE2_ADAPTER.iterdir()])

In [ ]:
files.download("stage2_adapter")

##Load inference model (no rules, pure model)

In [ ]:
# ── Pure model inference — zero rule-based logic ──────────────────────────
clear_memory()

infer_base, tokenizer = load_base_model()
infer_model = PeftModel.from_pretrained(infer_base, str(STAGE2_ADAPTER))
infer_model.eval()
print("Inference model ready.")


def generate_rewrite(question: str, topic: str, difficulty: str,
                     max_new_tokens: int = 80) -> str:
    """
    Rewrites `question` to `difficulty` ('harder' or 'easier').
    Pure model generation — no string manipulation, no templates.
    """
    prompt, _ = build_prompt_and_full_text(
        {"topic": topic, "base_question": question, difficulty: ""},
        difficulty,
    )
    # Remove the empty answer from full_text — we only need the prompt
    prompt = (
        f"### System:\n{SYSTEM_PROMPT}\n\n"
        f"### Input:\n"
        f"Topic: {topic}\n"
        f"Original Question: {question}\n"
        f"Target Difficulty: {difficulty}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(infer_model.device)
    with torch.no_grad():
        output = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (strip the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Remove any accidental "### ..." continuation
    result = re.split(r"###|Input:|System:", result)[0].strip()
    return result

##Test Evaluation

In [ ]:
from google.colab import files

# Upload test.json
files.upload()

with open("test.json") as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test questions")

# Run inference on all test questions
results = []

print("=" * 90)
print(f"{'ID':<6} {'TOPIC':<20} {'BASE QUESTION':<40}")
print("=" * 90)

for entry in test_data:
    topic        = entry["topic"]
    base_q       = entry["base_question"]
    expected_hard = entry.get("harder") or entry.get("medium", "")
    expected_easy = entry.get("easy", "")

    # Generate harder and easier from model
    generated_harder = generate_rewrite(base_q, topic, "harder")
    generated_easier = generate_rewrite(base_q, topic, "easier")

    print(f"\nID      : {entry['id']}")
    print(f"Topic   : {topic}")
    print(f"Base    : {base_q}")
    print(f"Expected Harder : {expected_hard}")
    print(f"Generated Harder: {generated_harder}")
    print(f"Expected Easier : {expected_easy}")
    print(f"Generated Easier: {generated_easier}")
    print("-" * 90)

    results.append({
        "id":               entry["id"],
        "topic":            topic,
        "base_question":    base_q,
        "expected_harder":  expected_hard,
        "generated_harder": generated_harder,
        "expected_easier":  expected_easy,
        "generated_easier": generated_easier,
    })

# Save results
with open("test_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"\nTotal questions evaluated: {len(results)}")
print("Results saved to test_results.json")

##Optional: zip and download the Stage 2 adapter

In [ ]:
import shutil
shutil.make_archive("stage2_adapter_final", "zip", str(STAGE2_ADAPTER))
files.download("stage2_adapter_final.zip")
print("Download started.")

NameError: name 'STAGE2_ADAPTER' is not defined

## RUN FROM HERE FOR HISTORY ONLY QUESTIONS

---

## New Section

--------------------------------------------------------------------------------------------------------------------------------------------------------------********************************************************************************

In [ ]:
!pip install -U "bitsandbytes>=0.46.1" "accelerate>=0.27.0" "transformers>=4.38.0" peft sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


In [ ]:
import bitsandbytes as bnb
import accelerate
import transformers

print("bitsandbytes:", bnb.__version__)
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)


bitsandbytes: 0.49.2
accelerate: 1.13.0
transformers: 5.5.4


In [ ]:
import gc, json, random, re, zipfile
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
random.seed(42)

BASE_MODEL     = "google/gemma-2b"
STAGE1_ZIP     = "gemma-math-domain-lora.zip"
STAGE1_DIR     = Path("stage1_adapter")
STAGE2_ZIP     = "stage2_gemma_lora.zip"
STAGE2_DIR     = Path("stage2_adapter")
STAGE3_ADAPTER = Path("stage3_adapter")
MAX_LEN        = 384

SYSTEM_PROMPT = (
    "You are a helpful curriculum assistant. "
    "Rewrite the given question to the requested difficulty level "
    "while keeping the topic and subject matter identical. "
    "Return only the rewritten question, nothing else."
)

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

print("Setup complete.")

Setup complete.


Cell upload for stage 1 adapters

In [ ]:
from google.colab import files
import shutil
import zipfile

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

stage1_uploaded = next((name for name in uploaded if "gemma-math-domain-lora" in name), None)
if stage1_uploaded is None:
    raise FileNotFoundError("Please upload the Stage 1 zip: gemma-math-domain-lora.zip")

if STAGE1_DIR.exists():
    shutil.rmtree(STAGE1_DIR)

STAGE1_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(stage1_uploaded, "r") as z:
    z.extractall(STAGE1_DIR)

print(f"Stage 1 unzipped to {STAGE1_DIR}")
print("Stage 1 contents:", [p.name for p in STAGE1_DIR.iterdir()])


Saving gemma-math-domain-lora.zip to gemma-math-domain-lora.zip
Uploaded files: ['gemma-math-domain-lora.zip']
Stage 1 unzipped to stage1_adapter
Stage 1 contents: ['tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'README.md', 'chat_template.jinja']


Cell for Stage 2

In [ ]:
from google.colab import files
import shutil
import zipfile

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

stage2_uploaded = next((name for name in uploaded if "stage2_gemma_lora" in name), None)
if stage2_uploaded is None:
    raise FileNotFoundError("Please upload the Stage 2 zip: stage2_gemma_lora.zip")

if STAGE2_DIR.exists():
    shutil.rmtree(STAGE2_DIR)

STAGE2_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(stage2_uploaded, "r") as z:
    z.extractall(STAGE2_DIR)

print(f"Stage 2 unzipped to {STAGE2_DIR}")
print("Stage 2 contents:", [p.name for p in STAGE2_DIR.iterdir()])


Saving stage2_gemma_lora.zip to stage2_gemma_lora.zip
Uploaded files: ['stage2_gemma_lora.zip']
Stage 2 unzipped to stage2_adapter
Stage 2 contents: ['tokenizer.json', 'tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'README.md']


Load base model + Stage 1 + Stage 2 (replace entirely):

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

login(token=HF_TOKEN)

def load_base_model():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        token=HF_TOKEN,
        quantization_config=bnb,
        device_map="auto",
    )
    return base, tokenizer


ithun changes kelet : trainingpn thodi badallli ahe , bas 1 subject specific

In [ ]:
from google.colab import files
import json

# Upload dataset
files.upload()

# Load dataset
with open("difficulty_dataset_combined.json") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data)} training examples")


# Build training examples (formatted)
FORMATTED_DATA = []

for entry in train_data:
    base = entry.get("input", "")
    output = entry.get("output", "")
    direction = entry.get("direction", "")

    if not base or not output or not direction:
        continue

    # Build prompt (input to model)
    prompt = (
        "Rewrite the equation based on direction.\n\n"
        f"Input: {base}\n"
        f"Direction: {direction}\n\n"
        "Output:"
    )

    # Target (what model should generate)
    target = f" {output}"

    FORMATTED_DATA.append({
        "prompt": prompt,
        "target": target
    })

print(f"Formatted training entries: {len(FORMATTED_DATA)}")


# Debug prints
from collections import Counter
directions = Counter(entry.get("direction", "") for entry in train_data)
print("Directions:", dict(directions))

print("\nSample formatted entry:")
print(FORMATTED_DATA[0])

Saving difficulty_dataset_combined.json to difficulty_dataset_combined.json
Loaded 2000 training examples
Formatted training entries: 2000
Directions: {'easier': 1000, 'harder': 1000}

Sample formatted entry:
{'prompt': 'Rewrite the equation based on direction.\n\nInput: 6x + 4 = 46\nDirection: easier\n\nOutput:', 'target': ' 6x = 42'}


Cell 5 — Tokenization

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "google/gemma-2b"  # or your replacement model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
import json

with open("difficulty_dataset_combined.json") as f:
    train_data = json.load(f)

NEW_EXAMPLES = []
for entry in train_data:
    base = entry.get("input", "")
    output = entry.get("output", "")
    direction = entry.get("direction", "")

    if not base or not output or not direction:
        continue

    NEW_EXAMPLES.append({
        "base_question": base,
        "modified_question": output,
        "direction": direction,
    })

print(f"Loaded {len(NEW_EXAMPLES)} examples")

Loaded 2000 examples


In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an expert at modifying equations to control their difficulty level.\n"
    "Given a base equation and a target difficulty (easier or harder), "
    "generate an appropriate modified version.\n"
    "- For 'easier': Simplify the equation, reduce complexity, make it more straightforward.\n"
    "- For 'harder': Add complexity, require more steps, incorporate additional terms."
)

MAX_LEN = 256

# ── Bridge: FORMATTED_DATA → tokenizer-ready records ─────────────────────────
# Cell 1 already built clean prompt+target pairs, so we use them directly.
# No need for NEW_EXAMPLES or a second prompt builder.

def tokenize_with_masking(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )["input_ids"]

    full_text = example["prompt"] + example["target"]

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

    labels = encoded["input_ids"].copy()
    prompt_len = min(len(prompt_ids), MAX_LEN)
    labels[:prompt_len] = [-100] * prompt_len          # mask prompt tokens
    labels = [
        t if m == 1 else -100
        for t, m in zip(labels, encoded["attention_mask"])
    ]
    encoded["labels"] = labels
    return encoded

raw_ds       = Dataset.from_list(FORMATTED_DATA)       # directly use Cell 1 output
tokenized_ds = raw_ds.map(tokenize_with_masking, remove_columns=raw_ds.column_names)

print(f"Tokenized dataset       : {tokenized_ds}")
print(f"Total records           : {len(tokenized_ds)}")
print(f"Non-masked label tokens : {sum(1 for l in tokenized_ds[0]['labels'] if l != -100)}")

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenized dataset       : Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})
Total records           : 2000
Non-masked label tokens : 34


In [ ]:
ids = tokenized_ds[0]["input_ids"]
mask = tokenized_ds[0]["attention_mask"]

tokens = tokenizer.convert_ids_to_tokens(
    [i for i, m in zip(ids, mask) if m == 1]
)

print(tokens)

['Rewrite', '▁the', '▁equation', '▁based', '▁on', '▁direction', '.', '\n\n', 'Input', ':', '▁', '6', 'x', '▁+', '▁', '4', '▁=', '▁', '4', '6', '\n', 'Direction', ':', '▁easier', '\n\n', 'Output', ':', '▁', '6', 'x', '▁=', '▁', '4', '2']


In [ ]:
print(tokenized_ds[0]["attention_mask"][:40])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


Cell 7 — Add Stage 3 LoRA and trai

In [ ]:
!pip install -U bitsandbytes>=0.46.1 transformers peft accelerate


In [ ]:
!pip install -U "bitsandbytes>=0.46.1" "accelerate>=0.27.0" "transformers>=4.38.0" peft sentencepiece

In [ ]:
!pip install -U "bitsandbytes>=0.46.1" "accelerate>=0.27.0" "transformers>=4.38.0" peft sentencepiece


In [ ]:
print("Stage 1 exists:", STAGE1_DIR.exists(), list(STAGE1_DIR.iterdir()))
print("Stage 2 exists:", STAGE2_DIR.exists(), list(STAGE2_DIR.iterdir()))



Stage 1 exists: True [PosixPath('stage1_adapter/tokenizer.json'), PosixPath('stage1_adapter/tokenizer_config.json'), PosixPath('stage1_adapter/adapter_model.safetensors'), PosixPath('stage1_adapter/adapter_config.json'), PosixPath('stage1_adapter/README.md'), PosixPath('stage1_adapter/chat_template.jinja')]
Stage 2 exists: True [PosixPath('stage2_adapter/tokenizer.json'), PosixPath('stage2_adapter/tokenizer_config.json'), PosixPath('stage2_adapter/adapter_model.safetensors'), PosixPath('stage2_adapter/adapter_config.json'), PosixPath('stage2_adapter/README.md')]


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RUN_DIR = "/content/drive/MyDrive/stage3_runs"


Mounted at /content/drive


In [ ]:
# stage3_args = TrainingArguments(
#     output_dir=DRIVE_RUN_DIR,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     num_train_epochs=3,
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.05,
#     fp16=True,
#     optim="paged_adamw_8bit",
#     logging_steps=20,
#     save_strategy="steps",
#     save_steps=200,          # adjust as needed
#     save_total_limit=2,      # keeps only recent checkpoints
#     report_to="none",
#     remove_unused_columns=False,
#     dataloader_pin_memory=False,
#     gradient_checkpointing=True,
#     max_grad_norm=0.3,
# )
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./difficulty-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,          # effective batch = 8, fine for Colab
    learning_rate=5e-5,                     # was 2e-4, too high for LoRA
    num_train_epochs=5,                     # was 3, small task benefits from more
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,                       # was 0.05, more warmup for stability
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=20,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,                      # was 0.3, too aggressive
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer

clear_memory()

# ------------------ LOAD BASE MODEL ------------------
base_model, tokenizer = load_base_model()

# 🔥 GEMMA FIXES
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# # ---- MERGE STAGE 1 ----
# print("Loading + merging Stage 1...")
# model = PeftModel.from_pretrained(base_model, str(STAGE1_DIR))
# model = model.merge_and_unload()

# # ---- MERGE STAGE 2 ----
# print("Loading + merging Stage 2...")
# model = PeftModel.from_pretrained(model, str(STAGE2_DIR))
# model = model.merge_and_unload()

# # ---- PREPARE FOR TRAINING ----
# model = prepare_model_for_kbit_training(model)
# ---- PREPARE FIRST (CRITICAL FIX) ----
model = prepare_model_for_kbit_training(base_model)

# ---- MERGE STAGE 1 ----
model = PeftModel.from_pretrained(model, str(STAGE1_DIR))
model = model.merge_and_unload()

# ---- MERGE STAGE 2 ----
model = PeftModel.from_pretrained(model, str(STAGE2_DIR))
model = model.merge_and_unload()
# model.gradient_checkpointing_enable()
model.config.use_cache = False

# ------------------ STAGE 3 LORA ------------------
lora3_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"   # 🔥 THIS FIXES YOUR PLATEAU
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

stage3_model = get_peft_model(model, lora3_config)
stage3_model.print_trainable_parameters()
stage3_model.train()

# ------------------ TRAINING ARGS ------------------
stage3_args = TrainingArguments(
    output_dir="/content/stage3_runs",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=5e-5,
    num_train_epochs=5,

    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    fp16=True,
    optim="paged_adamw_8bit",

    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,

    report_to="none",
    remove_unused_columns=False,

    dataloader_pin_memory=False,
    # gradient_checkpointing=True,

    max_grad_norm=1.0,
)

# ------------------ TRAINER ------------------
trainer = Trainer(
    model=stage3_model,
    args=stage3_args,
    train_dataset=tokenized_ds,
)

# ------------------ TRAIN ------------------
print("Starting Stage 3 training...")
trainer.train()
print("Stage 3 training complete.")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 19,611,648 || all params: 2,525,784,064 || trainable%: 0.7765
Starting Stage 3 training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
20,9.125363
40,6.813279
60,3.448374
80,1.375955
100,0.517037
120,0.381001
140,0.332595
160,0.311305
180,0.315493
200,0.311489


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


KeyboardInterrupt: 

In [ ]:
# # Save locally inside Colab
# final_save_path = "/content/mathfinal_model2"
# trainer.save_model(final_save_path)

# print("Model saved locally. Zipping...")

# # Zip the model folder
# import shutil
# shutil.make_archive("/content/mathfinal_model2", 'zip', final_save_path)

# print("Downloading...")

# from google.colab import files
# files.download("/content/mathfinal_model2.zip")
# ================= SAVE LORA =================
lora_path = "/content/mathfinal_model2_lora"
trainer.save_model(lora_path)

print("LoRA saved.")

# ================= SAVE MERGED MODEL =================
print("Merging model...")

merged_model = trainer.model.merge_and_unload()
full_path = "/content/mathfinal_model2_full"
merged_model.save_pretrained(full_path)

print("Merged model saved.")

# ================= ZIP FILES =================
import shutil

print("Zipping LoRA...")
shutil.make_archive("/content/mathfinal_model2_lora", 'zip', lora_path)

print("Zipping merged model...")
shutil.make_archive("/content/mathfinal_model2_full", 'zip', full_path)

# ================= DOWNLOAD =================
from google.colab import files

print("Downloading LoRA...")
files.download("/content/mathfinal_model2_lora.zip")

print("Downloading merged model...")
files.download("/content/mathfinal_model2_full.zip")

LoRA saved.
Merging model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved.
Zipping LoRA...
Zipping merged model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RUN FROM HERE FOR WORKING MATH ONLY:

upload model

In [ ]:
# ================= UPLOAD ZIP =================
from google.colab import files
uploaded = files.upload()

# ================= UNZIP =================
import zipfile
import os

zip_path = "/content/mathfinal_model2_lora.zip"
extract_path = "/content/mathfinal_model2_lora_extracted"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

# Fix nested folder issue
folders = os.listdir(extract_path)
if len(folders) == 1 and os.path.isdir(os.path.join(extract_path, folders[0])):
    lora_path = os.path.join(extract_path, folders[0])
else:
    lora_path = extract_path

print("LoRA path:", lora_path)

# ================= LOAD BASE MODEL =================
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model_name = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ================= LOAD LORA =================
model = PeftModel.from_pretrained(base_model, lora_path)

print("LoRA loaded successfully.")

# ================= TEST =================
prompt = "Rewrite the question to be harder: What is 2 + 2?"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Saving mathfinal_model2_lora.zip to mathfinal_model2_lora.zip
Extracted to: /content/mathfinal_model2_lora_extracted
LoRA path: /content/mathfinal_model2_lora_extracted


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


LoRA loaded successfully.
Rewrite the question to be harder: What is 2 + 2?

Answer:  

Step 1/3
2 + 2 = 4

Step 2/3
B: 4 = 2 + 2

Step 3/3
C: 4 = 2 + 2 + 2


In [ ]:
# from peft import PeftModel

# def load_trained_model(model_path):
#     base_model, tokenizer = load_base_model()

#     tokenizer.pad_token = tokenizer.eos_token
#     tokenizer.padding_side = "right"

#     model = PeftModel.from_pretrained(base_model, model_path)
#     model.eval()

#     return model, tokenizer

In [ ]:
# model, tokenizer = load_base_model()

# model = PeftModel.from_pretrained(
#     model,
#     "/content/mathfinal_model2_lora"   # ✅ correct path
# )
# model.eval()

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
         

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model = AutoModelForCausalLM.from_pretrained("/content/mathfinal_model2_full")
# tokenizer = AutoTokenizer.from_pretrained("/content/mathfinal_model2_full")

# model.eval()

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): GemmaTextScaledWordEmbedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
 

In [ ]:
def clean_answer(text):
    if "Output:" in text:
        text = text.split("Output:")[-1]

    # stop at next instruction or newline block
    text = text.split("\n")[0]

    return text.strip()

In [ ]:
def generate_clean(model, tokenizer, eq, direction):
    prompt = f"Input: {eq}\nDirection: {direction}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt").to(next(model.parameters()).device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return clean_answer(decoded)

In [ ]:
print(generate_clean(model, tokenizer, "4x + 6 = 18", "easier"))

x = 3


In [ ]:
mixed_tests = [
    # basic easier
    ("6x + 4 = 46", "easier"),
    ("8x + 3 = 35", "easier"),

    # harder
    ("6x + 4 = 46", "harder"),
    ("5x - 2 = 18", "harder"),

    # brackets
    ("3(x + 2) = 15", "easier"),
    ("4(x + 3) = 28", "easier"),
    ("5(x + 1) = 20", "harder"),

    # generalization
    ("10x + 15 = 65", "easier"),
    ("12x + 8 = 80", "easier"),

    # edge
    ("4x + 0 = 20", "easier"),
    ("1x + 5 = 12", "easier"),

    # out of distribution
    ("x/2 + 3 = 7", "easier"),
    ("3x^2 + 2 = 11", "easier"),
]

In [ ]:
import torch

def generate_with_fewshot(model, tokenizer, eq, direction):
    prompt = (
        "You transform equations based on difficulty.\n"
        "Rules:\n"
        "- easier: simplify equation\n"
        "- harder: add SAME number to BOTH sides\n"
        "- DO NOT solve for x\n"
        "- DO NOT expand brackets for harder\n"
        "- Output ONLY one equation\n\n"

        "Examples:\n"

        # ----- LINEAR -----
        "Input: 4x + 6 = 18\n"
        "Direction: harder\n"
        "Output: 4x + 6 + 3 = 18 + 3\n\n"

        "Input: 4x + 6 = 18\n"
        "Direction: easier\n"
        "Output: 4x = 12\n\n"

        # ----- BRACKETS (CRITICAL FIX) -----
        "Input: 3(x + 2) = 15\n"
        "Direction: harder\n"
        "Output: 3(x + 2) + 2 = 15 + 2\n\n"

        "Input: 4(x + 3) = 28\n"
        "Direction: harder\n"
        "Output: 4(x + 3) + 1 = 28 + 1\n\n"

        "Input: 3(x + 2) = 15\n"
        "Direction: easier\n"
        "Output: 3x + 6 = 15\n\n"

        # ----- TWO STEP -----
        "Input: 5x + 3 = 2x + 18\n"
        "Direction: harder\n"
        "Output: 5x + 3 + 4 = 2x + 18 + 4\n\n"

        "Input: 5x + 3 = 2x + 18\n"
        "Direction: easier\n"
        "Output: 3x + 3 = 18\n\n"

        # ----- TASK -----
        f"Input: {eq}\n"
        f"Direction: {direction}\n"
        "Output:"
    )

    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # extract clean output
    result = decoded.split("Output:")[-1].strip().split("\n")[0]

    return result


In [ ]:

import re

def is_expanded(eq):
    # detects if brackets disappeared
    return "(" not in eq and "x +" in eq


def generate_harder_strict(model, tokenizer, eq):
    # first attempt
    result = generate_with_fewshot(model, tokenizer, eq, "harder")

    # if model expanded → reject
    if "(" in eq and is_expanded(result):
        # retry with stronger constraint
        prompt = (
    "TASK: Apply the transformation.\n\n"

    f"DIRECTION: {direction.upper()}\n"
    f"EQUATION: {eq}\n\n"

    "Rules:\n"
    "- EASIER → simplify equation\n"
    "- HARDER → add SAME number to BOTH sides\n"
    "- Never mix them\n"
    "- Output ONLY the equation\n\n"

    "Examples:\n"

    "DIRECTION: HARDER\n"
    "EQUATION: 4x + 6 = 18\n"
    "OUTPUT: 4x + 6 + 2 = 18 + 2\n\n"

    "DIRECTION: EASIER\n"
    "EQUATION: 4x + 6 = 18\n"
    "OUTPUT: 4x = 12\n\n"

    "DIRECTION: HARDER\n"
    "EQUATION: 3(x + 2) = 15\n"
    "OUTPUT: 3(x + 2) + 1 = 15 + 1\n\n"

    "DIRECTION: EASIER\n"
    "EQUATION: 3(x + 2) = 15\n"
    "OUTPUT: 3x + 6 = 15\n\n"

    "NOW DO:\n"
    f"DIRECTION: {direction.upper()}\n"
    f"EQUATION: {eq}\n"
    "OUTPUT:"
)

        device = next(model.parameters()).device
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=30,
                do_sample=False,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        result = decoded.split("Output:")[-1].strip().split("\n")[0]

    return result


In [ ]:
tests = [
    ("6x + 4 = 46", "harder"),
    ("5x - 2 = 18", "harder"),
    ("3(x + 2) = 15", "harder"),
    ("3(x + 2) = 15", "easier"),
]

for eq, d in tests:
    print(eq, "→", generate_with_fewshot(model, tokenizer, eq, d))

6x + 4 = 46 → 6x + 4 + 1 = 46 + 1
5x - 2 = 18 → 5x - 2 + 2 = 18 + 2
3(x + 2) = 15 → 3x + 6 = 15
3(x + 2) = 15 → 3x + 6 = 15


In [ ]:
# ------------------ TEST ------------------

tests = [
    ("6x + 4 = 46", "easier"),
    ("5x - 2 = 18", "easier"),
    ("3(x + 2) = 15", "harder"),
    ("4(x + 3) = 28", "harder"),
]

for eq, d in tests:
    print(f"{eq} → {generate_with_fewshot(model, tokenizer, eq, d)}")

6x + 4 = 46 → 6x + 4 - 4 = 46 - 4
5x - 2 = 18 → 5x - 2 + 2 = 18 + 2
3(x + 2) = 15 → 3x + 6 = 15
4(x + 3) = 28 → 4x + 12 = 28


In [ ]:
import random
import re

def harder_rule(eq):
    k = random.randint(1, 5)

    if "=" not in eq:
        return eq

    left, right = eq.split("=")
    return f"{left.strip()} + {k} = {right.strip()} + {k}"


def smarter_generate(model, tokenizer, eq, direction):
    if direction == "harder":
        return harder_rule(eq)   # 🔥 bypass model

    else:
        # use model for easier
        return generate_clean(model, tokenizer, eq, direction)

In [ ]:
tests = [
    ("6x + 4 = 46", "easier"),
    ("5x - 2 = 18", "easier"),
    ("3(x + 2) = 15", "harder"),
    ("4(x + 3) = 28", "harder"),
]

for eq, d in tests:
    print(f"{eq} → {smarter_generate(model, tokenizer, eq, d)}")

6x + 4 = 46 → 6x = 42
5x - 2 = 18 → x = 6
3(x + 2) = 15 → 3(x + 2) + 1 = 15 + 1
4(x + 3) = 28 → 4(x + 3) + 1 = 28 + 1


In [ ]:
tests = [
    ("6x + 4 = 46", "harder"),
    ("5x - 2 = 18", "harder"),
    ("3(x + 2) = 15", "easier"),
    ("4(x + 3) = 28", "easier"),
]

for eq, d in tests:
    print(f"{eq} → {smarter_generate(model, tokenizer, eq, d)}")

6x + 4 = 46 → 6x + 4 + 3 = 46 + 3
5x - 2 = 18 → 5x - 2 + 2 = 18 + 2
3(x + 2) = 15 → x = 3
4(x + 3) = 28 → x = 7


In [ ]:
print(generate_clean(model, tokenizer, "6x + 4 = 46", "harder"))

6x = 42


In [ ]:
print(generate_clean(model, tokenizer, "6x + 4y = 46+y", "easier"))

6x = 46-y


working from here

##fallback

In [ ]:
import re

def is_harder_valid(original, generated):
    if "=" not in original or "=" not in generated:
        return False

    # must preserve original structure (not simplify)
    if "(" in original and "(" not in generated:
        return False  # brackets removed → easier

    # must add something to BOTH sides
    left_o, right_o = original.split("=")
    left_g, right_g = generated.split("=")

    return (
        left_g.strip() != left_o.strip() and
        right_g.strip() != right_o.strip() and
        "+" in left_g and "+" in right_g
    )


def is_easier_valid(original, generated):
    # easier should reduce complexity
    # simplest signal: fewer symbols or no brackets
    if "(" in original and "(" not in generated:
        return True

    # or removed constant pattern
    return generated.count("+") < original.count("+")

In [ ]:
import random

def fallback_make_harder(eq):
    k = random.randint(1, 199)

    if "=" not in eq:
        return eq

    left, right = eq.split("=")
    return f"{left.strip()} + {k} = {right.strip()} + {k}"

In [ ]:
def generate_with_validation(model, tokenizer, eq, direction):

    # Step 1: model generates
    candidate = generate_with_fewshot(model, tokenizer, eq, direction)

    # Step 2: validate
    if direction == "harder":
        if is_harder_valid(eq, candidate):
            return candidate
        else:
            # Step 3: fallback
            return fallback_make_harder(eq)

    elif direction == "easier":
        if is_easier_valid(eq, candidate):
            return candidate
        else:
            # optional fallback (keep model output if you prefer)
            return generate_clean(model, tokenizer, eq, direction)

    return candidate

In [ ]:
tests = [
    ("6x + 4 = 46", "easier"),
    ("5x - 2y = 18xy", "easier"),
    ("3(x + 2) = 15", "harder"),
    ("4(x + 3) = 28", "harder"),
]

for eq, d in tests:
    out = generate_with_validation(model, tokenizer, eq, d)
    print(f"{eq} → {out}")

6x + 4 = 46 → 6x = 42
5x - 2y = 18xy → 5x = 2y + 18
3(x + 2) = 15 → 3(x + 2) + 52 = 15 + 52
4(x + 3) = 28 → 4(x + 3) + 114 = 28 + 114


In [ ]:
tests = [
    ("3x + 5 = 46", "easier"),
    ("5x - 2y = 8xy", "easier"),
    ("4(x) = 15", "harder"),
    ("43(x + 3) = 28", "harder"),
]


for eq, d in tests:
    out = generate_with_validation(model, tokenizer, eq, d)
    print(f"{eq} → {out}")

3x + 5 = 46 → 3x = 41
5x - 2y = 8xy → 5x = 8xy + 2y
4(x) = 15 → 4(x) + 135 = 15 + 135
43(x + 3) = 28 → 43(x + 3) + 1 = 28 + 1


In [ ]:
!pip install nltk rouge-score bert-score -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bert_score

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
tests = [
    # ===== BASIC LINEAR =====
    ("2x + 4 = 10", "easier"),
    ("7x - 9 = 12", "easier"),
    ("3x + 5 = 20", "harder"),
    ("6x - 2 = 16", "harder"),

    # ===== MULTI-STEP =====
    ("2x + 3 = 11", "harder"),
    ("5x - 7 = 18", "harder"),
    ("4x + 12 = 20", "easier"),
    ("9x - 3 = 6", "easier"),

    # ===== BRACKETS =====
    ("2(x + 3) = 10", "easier"),
    ("3(x - 2) = 9", "easier"),
    ("4(x + 1) = 20", "harder"),
    ("5(x - 3) = 15", "harder"),

    # ===== FRACTIONS =====
    ("x/2 + 3 = 7", "easier"),
    ("x/3 + 5 = 11", "easier"),
    ("x/4 + 2 = 6", "harder"),
    ("x/5 + 1 = 3", "harder"),

    # ===== MULTI-VARIABLE =====
    ("2x + y = 10", "easier"),
    ("3x - y = 5", "easier"),
    ("x + y = 7", "harder"),
    ("2x - 3y = 8", "harder"),

    # ===== COMPLEX EXPRESSIONS =====
    ("3(x + 2) + 4 = 19", "easier"),
    ("2(x - 1) + 5 = 9", "easier"),
    ("4(x + 3) - 2 = 18", "harder"),
    ("5(x - 2) + 7 = 17", "harder"),

    # ===== EDGE CASES =====
    ("x = 5", "harder"),
    ("2x = 10", "easier"),
    ("x/2 = 4", "harder"),
    ("10 = 2x + 4", "easier"),

    # ===== SLIGHTLY TRICKIER =====
    ("2(x + 3) + x = 9", "easier"),
    ("3(x - 1) + 2x = 7", "easier"),
    ("x + 2(x + 3) = 12", "harder"),
    ("2x + 3(x - 1) = 11", "harder"),
]

In [ ]:
# ================= INSTALL =================
!pip install bert-score -q

# ================= IMPORTS =================
from bert_score import score

# ================= GENERATE + CLEAN =================
inputs = []
preds = []

def is_valid_output(out):
    return (
        out is not None and
        isinstance(out, str) and
        "=" in out and
        len(out.strip()) > 3
    )

print("Generating outputs...\n")

for eq, d in tests:
    out = generate_with_validation(model, tokenizer, eq, d)

    print(f"{eq} → {out}")

    if is_valid_output(out):
        inputs.append(eq)
        preds.append(out)

print("\nValid samples:", len(preds), "/", len(tests))

# ================= SAFETY CHECK =================
if len(preds) == 0:
    print("No valid outputs. Cannot compute BERTScore.")
else:
    # ================= BERTSCORE =================
    print("\nComputing BERTScore...\n")

    P, R, F1 = score(
        preds,
        inputs,
        model_type="bert-base-uncased",   # FIXED (prevents your error)
        lang="en",
        verbose=True
    )

    avg_p = P.mean().item()
    avg_r = R.mean().item()
    avg_f1 = F1.mean().item()

    print("\n===== BERTScore Results =====")
    print(f"Precision: {avg_p:.4f}")
    print(f"Recall   : {avg_r:.4f}")
    print(f"F1       : {avg_f1:.4f}")

    # ================= SEMANTIC CHANGE =================
    changes = [1 - f.item() for f in F1]
    avg_change = sum(changes) / len(changes)

    print("\n===== Transformation Strength =====")
    print(f"Avg Semantic Change: {avg_change:.4f}")

# ================= BASIC QUALITY STATS =================
total = len(tests)
valid = len(preds)

print("\n===== Basic Stats =====")
print(f"Valid Output %: {valid/total:.2f}")

Generating outputs...

2x + 4 = 10 → 2x = 6
7x - 9 = 12 → 7x = 21
3x + 5 = 20 → 3x + 5 + 3 = 20 + 3
6x - 2 = 16 → 6x - 2 + 2 = 16 + 2
2x + 3 = 11 → 2x + 3 + 1 = 11 + 1
5x - 7 = 18 → 5x - 7 + 7 = 18 + 7
4x + 12 = 20 → 4x = 8
9x - 3 = 6 → 9x =
2(x + 3) = 10 → 2x + 6 = 10
3(x - 2) = 9 → 
4(x + 1) = 20 → 4(x + 1) + 1 = 20 + 1
5(x - 3) = 15 → 5(x - 3) + 1 = 15 + 1
x/2 + 3 = 7 → x = 14
x/3 + 5 = 11 → x = 28
x/4 + 2 = 6 → x/4 + 2 + 4 = 6 + 4
x/5 + 1 = 3 → x/5 + 1 + 4 = 3 + 4
2x + y = 10 → 2x + y = 10
3x - y = 5 → 3x = 5 + y
x + y = 7 → x + y + 160 = 7 + 160
2x - 3y = 8 → 2x - 3y + 3 = 8 + 3
3(x + 2) + 4 = 19 → 3x + 8 = 19
2(x - 1) + 5 = 9 → 2x - 2 + 5 = 9 + 5
4(x + 3) - 2 = 18 → 4(x + 3) - 2 + 2 = 18 + 2
5(x - 2) + 7 = 17 → 5(x - 2) + 7 + 1 = 17 + 1
x = 5 → x + 138 = 5 + 138
2x = 10 → 5
x/2 = 4 → x/2 + 19 = 4 + 19
10 = 2x + 4 → 14 = 2x + 4
2(x + 3) + x = 9 → 3x + 6 = 9
3(x - 1) + 2x = 7 → 3x - 3 + 2x = 7
x + 2(x + 3) = 12 → x + 2(x + 3) + 1 = 12 + 1
2x + 3(x - 1) = 11 → 2x + 3(x - 1) + 118 = 

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.15 seconds, 204.33 sentences/sec

===== BERTScore Results =====
Precision: 0.8343
Recall   : 0.8355
F1       : 0.8319

===== Transformation Strength =====
Avg Semantic Change: 0.1681

===== Basic Stats =====
Valid Output %: 0.94


import random

def fallback_make_harder(eq):
    k = random.randint(1, 5)
    left, right = eq.split("=")
    return f"{left.strip()} + {k} = {right.strip()} + {k}"


def generate_with_fallback_flag(model, tokenizer, eq, direction):
    candidate = generate_with_fewshot(model, tokenizer, eq, direction)

    fallback_used = False

    if direction == "harder":
        if not is_harder_valid(eq, candidate):
            candidate = fallback_make_harder(eq)
            fallback_used = True

    elif direction == "easier":
        if not is_easier_valid(eq, candidate):
            fallback_used = True  # optional, if you retry or enforce

    return candidate, fallback_used

tests = [
    ("6x + 4 = 46", "easier"),
    ("5x - 2 = 18", "easier"),
    ("3(x + 2) = 15", "harder"),
    ("4(x + 3) = 28", "harder"),
]

for eq, d in tests:
    result, used = generate_with_fallback_flag(model, tokenizer, eq, d)

    print(f"\nInput     : {eq}")
    print(f"Direction : {d}")
    print(f"Output    : {result}")
    print(f"Fallback  : {'YES' if used else 'NO'}")

WORKING TILL HERE